# Functions and Arguments

## Quick notes

### What this covers
- Defining functions (two syntaxes), calling them, and how arguments arrive as `$1 $2 ... $#`.
- The three ways to hand data back: exit status (`return`), printing (`echo` + `$(...)`), and setting a variable by name.
- `"$@"` vs `"$*"` vs unquoted `$@`, `shift`, default values, `local`, recursion, pipelines, and small function libraries.

### Key points
- A bash function behaves like a tiny script: it has its own `$1`, `$2`, `$#`, `$@`, but shares the rest of the shell (variables, cwd, other functions).
- `return N` only sets the exit status (0-255). Data goes out through `echo`/`printf` (captured with `$(...)`) or through a variable.
- Always write `"$@"` (with quotes) to pass arguments along unchanged.
- Use `local` for every variable you do not mean to share.

### Gotchas (the part worth revising twice)
- Unquoted `$@` and `$*` split arguments that contain spaces. Only `"$@"` keeps each argument whole.
- `$(f)` runs `f` in a subshell: variables it sets are lost. A function in a pipeline is a subshell too.
- Variables are global by default. Bash `local` is dynamically scoped: callees see the caller's locals.
- `local x=$(cmd)` hides the exit status of `cmd` (you get the status of `local`).
- `return` outside a function (or a sourced file) is an error. `exit` inside a function kills the whole shell.

### When it matters
- Any script longer than ~20 lines: functions remove repetition and make error handling testable.
- Libraries you `source` from several scripts.

### One-line recap example
```bash
add() { local sum=$(( $1 + $2 )); echo "$sum"; }; total=$(add 3 4)   # total=7
```

Related: `01-Basics/01-Fundamentals` (quoting, expansion), `01-Basics/02-Control-Flow` (exit statuses, `if`), `01-Basics/04-Strings-And-Arrays` (arrays), `04-Scripting-Craft/01-Advanced-Bash` (`set -e`, traps and other advanced features).

## 01. Defining and calling functions
- Two syntaxes: `name() { ...; }` (POSIX, portable) and `function name { ...; }` (bash/ksh). Prefer the POSIX form.
- When you call a function you write no parentheses: `greet Ada`.
- The body must end with `;` or a newline before the closing `}`, and there must be a space after `{`.
- A function must be defined before it is called (at run time, not in file order).

In [1]:
# setup: a %%bash cell magic that keeps ONE bash session alive between cells
import atexit, os, shutil, subprocess, tempfile, uuid
from IPython.core.magic import register_cell_magic

_WORK = tempfile.mkdtemp(prefix="bashnb_")
atexit.register(shutil.rmtree, _WORK, ignore_errors=True)
_ENV = {k: v for k, v in os.environ.items() if k not in ("CLICOLOR", "CLICOLOR_FORCE", "FORCE_COLOR", "PYDEVD_DISABLE_FILE_VALIDATION")}
_ENV.update(LC_ALL="C", TZ="UTC", HOME=_WORK, PS1="", PS4="+ ")
_session = None

def _start():
    global _session
    _session = subprocess.Popen(["bash", "--noprofile", "--norc"], stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=_WORK, env=_ENV)

def _clean(text):
    return text.replace(os.path.join(_WORK, "script"), "script").replace(_WORK, "<work>")

def _run_persistent(code, timeout=60):
    global _session
    if _session is None or _session.poll() is not None:
        _start()
    token = uuid.uuid4().hex
    path = os.path.join(_WORK, "script")
    with open(path, "w") as fh:
        fh.write(code + "\n")
    _session.stdin.write(". '%s' </dev/null; printf '\\n%s %%d\\n' $?\n" % (path, token))
    _session.stdin.flush()
    out, status = [], None
    import threading
    result = {}
    def reader():
        while True:
            line = _session.stdout.readline()
            if not line:
                result["status"] = None; return
            if line.startswith(token + " "):
                result["status"] = int(line.split()[1]); return
            out.append(line)
    t = threading.Thread(target=reader, daemon=True); t.start(); t.join(timeout)
    os.remove(path)
    if t.is_alive():
        _session.kill(); _session = None
        return "".join(out) + "[timed out]\n", 124
    if result.get("status") is None:       # the shell itself exited (e.g. the cell ran `exit N`)
        code = _session.wait(); _session = None
        return "".join(out) + "(the bash session ended; the next cell starts a fresh one)\n", code
    text = "".join(out)
    if text.endswith("\n\n"):
        text = text[:-1]          # drop the blank line our status marker adds
    return text, result.get("status") if result.get("status") is not None else 1

def _run_fresh(code, timeout=60):
    path = os.path.join(_WORK, "script")
    with open(path, "w") as fh:
        fh.write(code + "\n")
    done = subprocess.run(["bash", "--noprofile", "--norc", path], stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, cwd=_WORK, env=_ENV, timeout=timeout)
    os.remove(path)
    return done.stdout, done.returncode

@register_cell_magic
def bash(line, cell):
    """Run the cell in bash. Flags: fresh (new shell), expect_fail (show non-zero status instead of raising)."""
    flags = line.split()
    text, status = (_run_fresh if "fresh" in flags else _run_persistent)(cell)
    print(_clean(text), end="" if text.endswith("\n") or not text else "\n")
    if status != 0:
        if "expect_fail" in flags:
            print("[exit status %d]" % status)
        else:
            raise RuntimeError("bash cell exited with status %d" % status)

In [2]:
%%bash
greet() { echo "Hello, $1!"; }
function shout { echo "HEY, $1!"; }
greet Ada
shout Linus

Hello, Ada!
HEY, Linus!


In [3]:
%%bash expect_fail
# Calling before defining fails, because definitions are executed like commands.
later_defined_caller() { not_yet_defined; }
later_defined_caller
not_yet_defined() { echo "now I exist"; }
later_defined_caller

script: line 2: not_yet_defined: command not found
now I exist


## 02. Positional parameters
- Inside a function `$1 ... $9`, `${10}`, `$#`, `$@`, `$*` refer to the function's own arguments, not the script's.
- `$0` is still the script name (not the function name). Use `${FUNCNAME[0]}` for the function name.
- Above 9 you need braces: `${10}`; `$10` means `${1}` followed by `0`.

In [4]:
%%bash
show() {
  echo "name=${FUNCNAME[0]} count=$# first=[$1] second=[$2] tenth=[${10}]"
  echo "dollar-10 is: [$10]  (that is \$1 followed by a literal 0)"
}
show a b c d e f g h i j k
show

name=show count=11 first=[a] second=[b] tenth=[j]
dollar-10 is: [a0]  (that is $1 followed by a literal 0)
name=show count=0 first=[] second=[] tenth=[]
dollar-10 is: [0]  (that is $1 followed by a literal 0)


## 03. `"$@"` vs `"$*"` vs unquoted `$@`
- `"$@"` gives one word per argument, exactly as passed. This is what you want almost always.
- `"$*"` joins everything into a single word, separated by the first character of `IFS`.
- Unquoted `$@` and `$*` are re-split on whitespace (and globbed!), so `"two words"` becomes two words.
- The helper below prints each word it receives inside `<>` so the difference is visible.

In [5]:
%%bash
args() { printf 'count=%d:' "$#"; for a in "$@"; do printf ' <%s>' "$a"; done; echo; }
f() {
  echo '--- "$@"';  args "$@"
  echo '--- "$*"';  args "$*"
  echo '--- $@';    args $@
  echo '--- $*';    args $*
}
f "hello world" second "third  with   spaces"

--- "$@"
count=3: <hello world> <second> <third  with   spaces>
--- "$*"
count=1: <hello world second third  with   spaces>
--- $@
count=6: <hello> <world> <second> <third> <with> <spaces>
--- $*
count=6: <hello> <world> <second> <third> <with> <spaces>


In [6]:
%%bash
# "$*" uses the first character of IFS as the separator, which is handy for joining.
join_with() { local IFS="$1"; shift; echo "$*"; }
join_with , a b c
join_with ' | ' x y z       # only the FIRST char of IFS is used

a,b,c
x y z


## 04. Counting and looping over arguments
- `$#` is the number of arguments. `for a in "$@"` is the idiomatic loop; plain `for a; do` means the same thing.
- Check argument counts early and fail with a message on stderr plus a non-zero status.

In [7]:
%%bash
need_two() {
  if [ "$#" -ne 2 ]; then
    echo "usage: need_two A B (got $#)" >&2
    return 2
  fi
  echo "ok: $1 and $2"
}
need_two only-one
echo "status=$?"
need_two x "y z"
echo "status=$?"
for_demo() { for a; do echo "arg: $a"; done; }   # "for a; do" == for a in "$@"
for_demo "one two" three

usage: need_two A B (got 1)
status=2
ok: x and y z
status=0
arg: one two
arg: three


## 05. `shift` - consuming arguments
- `shift` drops `$1` and moves the rest down (`$2` becomes `$1`). `shift N` drops N.
- `shift` with nothing left is an error (status 1, and in some shells it aborts); check `$#` first.
- Classic use: peel off options or a command word, then pass the rest along with `"$@"`.

In [8]:
%%bash
sum_all() {
  local total=0
  while [ "$#" -gt 0 ]; do
    total=$(( total + $1 ))
    shift
  done
  echo "$total"
}
sum_all 1 2 3 4
run_logged() {
  local label=$1; shift          # first arg is a label, the rest is a command
  echo "[$label] running: $*"
  "$@"
}
run_logged demo printf '%s\n' "a b" c

10
[demo] running: printf %s\n a b c
a b
c


In [9]:
%%bash expect_fail
# shift past the end: returns non-zero (and in bash prints an error only if shopt shift_verbose is on)
f() { shift 3; }
f a b


[exit status 1]


## 06. Default values and checks: `:-` `:=` `:+` `:?`
- `${v:-x}` use `x` if `v` is unset or empty (does not assign).
- `${v:=x}` same, but also assigns `x` to `v`. (It cannot assign to positional parameters such as `$1`.)
- `${v:+x}` use `x` only if `v` is set and non-empty (else empty). Good for optional flags.
- `${v:?message}` abort with `message` if `v` is unset or empty. A non-interactive shell EXITS, so the demos below run in subshells `( ... )`.
- Without the colon (`${v-x}`) only *unset* counts, not empty.

In [10]:
%%bash
f() {
  local name=${1:-world}
  echo "hello, $name"
}
f; f Ada; f ""
unset v; echo "[${v:-fallback}] still unset? [${v-unset}]"
echo "[${v:=assigned}] now v=[$v]"
flag=; echo "[${flag:+--verbose}] [${flag-notset}]"
flag=1; echo "[${flag:+--verbose}]"
empty=; echo "[${empty-dash}] [${empty:-colon-dash}]"

hello, world
hello, Ada
hello, world
[fallback] still unset? [unset]
[assigned] now v=[assigned]
[] []
[--verbose]
[] [colon-dash]


In [11]:
%%bash expect_fail
need() { local f=${1:?need a file name}; echo "got $f"; }
( need data.txt )
( need )            # the subshell exits; the message goes to stderr
echo "outer shell alive; the subshell exited with status $?"

got data.txt
script: line 1: 1: need a file name
outer shell alive; the subshell exited with status 1


## 07. Returning data, way 1: exit status with `return`
- `return N` (0-255) sets `$?`. Zero means success/true, anything else means failure/false.
- Without `return`, the status is that of the last command run.
- This makes functions usable directly in `if`, `&&`, `||`, `while`.
- `return` with no number returns the status of the last command (`return $?`).

In [12]:
%%bash
is_even() { [ $(( $1 % 2 )) -eq 0 ]; }       # status of the test IS the return value
for n in 3 4; do
  if is_even "$n"; then echo "$n even"; else echo "$n odd"; fi
done
check_file() {
  [ -f "$1" ] || { echo "missing: $1" >&2; return 3; }
  return 0
}
check_file nope.txt; echo "status=$?"
big() { return 300; }; big; echo "return 300 wraps to: $?"

3 odd
4 even
missing: nope.txt
status=3
return 300 wraps to: 44


## 08. Returning data, way 2: print it and capture with `$(...)`
- The most common way to "return a string or number".
- Capture with `result=$(func args)`. Trailing newlines are stripped.
- Cost: a subshell (a fork). Anything the function sets (variables, `cd`) is lost.
- Keep diagnostics on stderr (`>&2`) so they are not captured as the result.

In [13]:
%%bash
add() { echo $(( $1 + $2 )); }
sum=$(add 3 5)
echo "3+5=$sum"
noisy() { echo "debug: working" >&2; echo "RESULT"; }
r=$(noisy 2>/dev/null)
echo "captured only: [$r]"
counter=0
bump() { counter=$((counter + 1)); echo "bumped"; }
x=$(bump)
echo "after \$(bump): counter=$counter (the increment happened in a subshell)"
bump >/dev/null
echo "after plain call: counter=$counter"

3+5=8
captured only: [RESULT]
after $(bump): counter=0 (the increment happened in a subshell)
after plain call: counter=1


## 09. Returning data, way 3: a global variable or an out-parameter by name
- Avoid the subshell by writing to a variable. Simplest: set a well-known global, e.g. `REPLY`-style.
- Better: the caller passes the *name* of the variable to fill. `printf -v name fmt args` assigns without `eval` (bash 3.1+, verified on 3.2 below).
- With `eval` you can do the same but must quote carefully, since the value goes through the parser twice. Only use it with variable names written in your own code, never with names or values taken from input.
- Namerefs (`local -n out=$1`) are cleaner, but need bash 4.3+. They are shown in a non-run block at the end of this section.
- Name clash: if the function has a `local` with the same name as the caller's variable, the assignment hits the local. Use unlikely local names (`_out`).

In [14]:
%%bash
set_upper() {                       # set_upper VARNAME text...
  local _out=$1; shift
  local _val; _val=$(echo "$*" | tr '[:lower:]' '[:upper:]')
  printf -v "$_out" '%s' "$_val"
}
set_upper result "hello there"
echo "result=[$result]"
printf -v padded '%-6s|%05d' ab 42
echo "[$padded]"

result=[HELLO THERE]
[ab    |00042]


In [15]:
%%bash
# eval version: works everywhere, but the quoting is easy to get wrong
set_len() { eval "$1=\${#2}"; }
set_len n "twelve chars"
echo "n=$n"
# The name-clash trap: the function's own local shadows the caller's variable
bad() { local out; printf -v "$1" '%s' "value"; }
out=original; bad out; echo "caller's out is still [$out] (the local ate the write)"

n=12
caller's out is still [original] (the local ate the write)


Bash 4.3+ nameref version (not run here, this machine has bash 3.2):

```bash
# needs bash 4.3+ (not run here)
set_upper() { local -n out=$1; out=${2^^}; }
set_upper result "hello"; echo "$result"      # HELLO
```

## 10. `local` scoping, and the dynamic scope gotcha
- Without `local`, an assignment inside a function creates or overwrites a global.
- `local x` makes `x` private to the call, and shadows any global of the same name until the function returns.
- Bash uses *dynamic* scope: a function called from another function sees the caller's locals (unlike lexical scoping in most languages). `inner` below sees `outer`'s `x`.
- `local x=$(cmd)` returns the status of `local` (always 0), hiding failure of `cmd`. Split it: `local x; x=$(cmd)`.

In [16]:
%%bash
x=global
leaky() { x=changed; y=leaked; }
leaky
echo "x=$x y=$y"
x=global; unset y
safe() { local x=private; local y=private; echo "inside: x=$x y=$y"; }
safe
echo "outside: x=$x y=${y-unset}"

x=changed y=leaked
inside: x=private y=private
outside: x=global y=unset


In [17]:
%%bash
inner() { echo "inner sees x=$x"; x=modified-by-inner; }
outer() { local x=outers-local; inner; echo "outer's local is now: $x"; }
x=global
outer
echo "global x untouched: $x"

inner sees x=outers-local
outer's local is now: modified-by-inner
global x untouched: global


In [18]:
%%bash
# the exit-status trap with local
try() { local out=$(false); echo "status after local x=\$(false): $?"; }
try
try2() { local out; out=$(false); echo "status after split form: $?"; }
try2

status after local x=$(false): 0
status after split form: 1


## 11. Recursion
- Functions can call themselves. Always give a base case.
- Locals make each call independent. Bash has no tail-call optimisation and the stack is limited, so keep recursion shallow; a loop is usually better.
- Return values come through `$(...)` (a subshell per call, so it is slow) or a global.

In [19]:
%%bash
fact() {
  local n=$1
  if [ "$n" -le 1 ]; then echo 1; return; fi
  local prev; prev=$(fact $((n - 1)))
  echo $(( n * prev ))
}
fact 5
fib() { local n=$1; if (( n < 2 )); then echo "$n"; else echo $(( $(fib $((n-1))) + $(fib $((n-2))) )); fi; }
fib 10
# not recursive: just shows `printf '%*s'` indenting items by depth
tree() { local depth=$1; shift; local item; for item in "$@"; do printf '%*s%s\n' $((depth*2)) '' "$item"; done; }
tree 0 root; tree 1 a b; tree 2 leaf

120
55
root
  a
  b
    leaf


## 12. Functions in pipelines (they run in subshells)
- A function can be a stage in a pipeline and reads stdin like any command.
- But every stage of a pipeline runs in its own subshell (in bash 3.2 and by default in later versions), so variables set there vanish.
- Fix: use process substitution `< <(cmd)` or redirect from a file/here-string so the loop stays in the current shell.

In [20]:
%%bash
upper_lines() { while IFS= read -r line; do printf '%s\n' "$line" | tr '[:lower:]' '[:upper:]'; done; }
printf 'one\ntwo\n' | upper_lines
total=0
sum_stdin() { while read -r n; do total=$((total + n)); done; echo "inside: total=$total"; }
printf '1\n2\n3\n' | sum_stdin
echo "outside: total=$total  (the pipeline stage was a subshell)"
total=0
sum_stdin < <(printf '1\n2\n3\n')
echo "outside after redirect: total=$total"

ONE
TWO
inside: total=6
outside: total=0  (the pipeline stage was a subshell)
inside: total=6
outside after redirect: total=6


## 13. Exit status propagation
- A function's status is the status of its last command, or the argument to `return`.
- `f | g` has the status of `g`, unless `set -o pipefail` is on. `PIPESTATUS` holds each stage.
- `if f; then ...` treats `f` as a condition; with `set -e`, failures inside `if` conditions, `&&`/`||` lists are ignored. (Details in `04-Scripting-Craft/01-Advanced-Bash`.)
- Capturing: `out=$(f); rc=$?` gives you both the output and the status.

In [21]:
%%bash
f() { echo "doing work"; return 7; }
out=$(f); rc=$?
echo "out=[$out] rc=$rc"
last_wins() { false; true; }        # earlier failures do not count
last_wins; echo "last_wins status=$?"
true | false | true; echo "pipeline status=$? PIPESTATUS=(${PIPESTATUS[*]})"
false | true; echo "plain: $?"
( set -o pipefail; false | true; echo "pipefail: $?" )

out=[doing work] rc=7
last_wins status=0
pipeline status=0 PIPESTATUS=(0 1 0)
plain: 0
pipefail: 1


In [22]:
%%bash fresh expect_fail
# exit inside a function ends the whole shell, not just the function
f() { echo "in f"; exit 4; }
f
echo "never reached"

in f
[exit status 4]


## 14. Inspecting functions: `declare -f`, `type`, `unset -f`
- `declare -f name` prints the function's source (as bash re-formats it). `declare -F` lists names only.
- `type name` says whether a word is a function, builtin, alias or file. `type -t name` gives just the kind.
- `unset -f name` removes a function. A function shadows a command of the same name; use `command name` to bypass it.

In [23]:
%%bash
hello() { echo "hello $1"; }
declare -f hello
type hello | head -1
echo "kind: $(type -t hello) / $(type -t cd) / $(type -t ls)"
ls() { echo "my ls function"; }
ls
command ls >/dev/null && echo "command ls reached the real ls"
unset -f ls
type -t ls
unset -f hello; type -t hello || echo "hello is gone (status $?)"

hello () 
{ 
    echo "hello $1"
}
hello is a function
kind: function / builtin / file
my ls function
command ls reached the real ls
file
hello is gone (status 1)


## 15. Naming and exporting functions
- Function names live in their own namespace, separate from variables (`f` and `f=1` coexist).
- Use lowercase words with underscores. A library prefix (`mylib_check`) avoids clashes. Avoid names of builtins and common commands.
- Functions are not inherited by child scripts, unless exported with `export -f name` (a bash-only feature: it passes the function in the environment). `bash -c` children then see it.
- Exported functions are a long-standing security-sensitive feature (Shellshock). Prefer sourcing a library in the child.

In [24]:
%%bash
helper() { echo "helper called with [$*]"; }
bash -c 'helper one' 2>&1 | head -1
export -f helper
bash -c 'helper two'
export -nf helper                       # stop exporting
bash -c 'helper three' 2>&1 | head -1
f=1; f() { echo "f the function, f the variable is $f"; }; f

bash: helper: command not found
helper called with [two]
bash: helper: command not found
f the function, f the variable is 1


## 16. Passing arrays to functions and by name
- Arrays cannot be passed as one argument. Expand them: `f "${arr[@]}"` and receive them with `"$@"`.
- To pass several arrays, or to modify one in place, pass the *name* and use `eval` (3.2-safe), or indirect expansion `${!ref}` with `ref="name[@]"`.
- A function cannot return an array. Print one element per line and read it back, or fill a named variable.
- Full array syntax is in `01-Basics/04-Strings-And-Arrays`.

In [25]:
%%bash
sum_arr() { local s=0 n; for n in "$@"; do s=$((s + n)); done; echo "$s"; }
nums=(3 4 5)
sum_arr "${nums[@]}"
names=("Ada Lovelace" "Alan Turing")
count_args() { echo "$# arguments"; }
count_args "${names[@]}"       # 2, spaces preserved
count_args ${names[@]}         # 4 (bug): unquoted splits the words

12
2 arguments
4 arguments


In [26]:
%%bash
# by name with indirect expansion: ref holds the text "name[@]"
print_named() { local ref="$1[@]"; local item; for item in "${!ref}"; do echo "- $item"; done; }
print_named names
# append to a caller's array by name with eval
append_to() { eval "$1+=(\"\$2\")"; }
append_to names "Grace Hopper"
echo "now ${#names[@]} names: ${names[2]}"
# "return" an array by filling a named variable from lines
make_list() { printf '%s\n' alpha beta gamma; }
out=(); while IFS= read -r l; do out+=("$l"); done < <(make_list)
echo "got ${#out[@]}: ${out[*]}"

- Ada Lovelace
- Alan Turing
now 3 names: Grace Hopper
got 3: alpha beta gamma


## 17. Error handling inside functions
- Return non-zero for failure and send messages to stderr. Let the caller decide what to do (exit, retry, ignore).
- Check the status right away, because `$?` is overwritten by the next command.
- `||` with a group is a compact guard: `cmd || { echo "failed" >&2; return 1; }`.
- Never `exit` from a library function unless the whole point is to kill the script; `return` is polite.
- A `die` helper is the one place where `exit` belongs.

In [27]:
%%bash
die() { echo "error: $*" >&2; return 1; }     # in a script you'd use exit; return keeps this demo alive
read_config() {
  [ -r "$1" ] || { die "cannot read $1"; return 1; }
  echo "reading $1"
}
read_config missing.conf; echo "status=$?"
printf 'k=v\n' > ok.conf
read_config ok.conf && echo "success path"
# $? is overwritten by the very next command
false; echo "saw $?"; echo "now $?"
rm -f ok.conf

error: cannot read missing.conf
status=1
reading ok.conf
success path
saw 1
now 0


In [28]:
%%bash fresh expect_fail
# with set -e, a failing command inside a function stops the script, but not when tested by if/||/&&
set -e
step() { false; echo "step continued"; }
if step; then echo "(inside if: set -e is ignored, so step continued)"; fi
step || echo "(inside ||: set -e is ignored too)"
step
echo "never reached"

step continued
(inside if: set -e is ignored, so step continued)
step continued
[exit status 1]


## 18. Function libraries with `source`
- Put reusable functions in a file and load it with `source lib.sh` (or `. lib.sh`, the POSIX spelling).
- Sourced code runs in the current shell, so its functions and variables become yours. `return` works at the top level of a sourced file.
- Add an include guard so double-sourcing is harmless. Give functions a prefix to avoid collisions.
- A library should define things and do nothing else (no output, no `exit`, no `set -e`, which would change the caller's shell).
- `source` searches `PATH` for names without a slash. Use `source ./lib.sh` for files in the current directory.

In [29]:
%%bash
cat > mylib.sh <<'LIB'
# mylib.sh - sourced, not executed
[ -n "${MYLIB_LOADED-}" ] && return 0
MYLIB_LOADED=1
mylib_version=1.0
mylib_trim() { local s=$1; s=${s#"${s%%[![:space:]]*}"}; s=${s%"${s##*[![:space:]]}"}; printf '%s' "$s"; }
mylib_repeat() { local i; for ((i = 0; i < $2; i++)); do printf '%s' "$1"; done; echo; }
LIB
source ./mylib.sh
source ./mylib.sh        # second load returns early because of the guard
echo "loaded=$MYLIB_LOADED version=$mylib_version"
echo "[$(mylib_trim '   padded text  ')]"
mylib_repeat ab 3

loaded=1 version=1.0
[padded text]
ababab


In [30]:
%%bash
# $0 and BASH_SOURCE: inside a sourced file they differ
cat > who.sh <<'LIB'
whoami_src() { echo "BASH_SOURCE[0]=${BASH_SOURCE[0]}"; }
LIB
. ./who.sh
whoami_src
rm -f who.sh

BASH_SOURCE[0]=./who.sh


## 19. A taste of `getopts` (full treatment later)
- `getopts "ab:" opt` parses `-a` (flag) and `-b value` (option with argument) one per call, filling `$opt` and `$OPTARG`.
- Use it in a `while` loop; afterwards `shift $((OPTIND - 1))` leaves only the non-option arguments.
- In a *function*, reset `OPTIND=1` first (and make it `local`), otherwise the second call starts where the first stopped.
- Only short options. `getopts` does not handle long options (`--name`); parse those by hand with a `case` loop.

In [31]:
%%bash expect_fail
parse() {
  local OPTIND=1 opt verbose=0 name=default
  while getopts "vn:" opt; do
    case $opt in
      v) verbose=1 ;;
      n) name=$OPTARG ;;
      *) echo "usage: parse [-v] [-n name] files..." >&2; return 2 ;;
    esac
  done
  shift $((OPTIND - 1))
  echo "verbose=$verbose name=$name rest=[$*]"
}
parse -v -n Ada one two
parse -n "Ada L" file
parse -x 2>&1

verbose=1 name=Ada rest=[one two]
verbose=0 name=Ada L rest=[file]
bash: illegal option -- x
usage: parse [-v] [-n name] files...
[exit status 2]


## 20. A small, testable function library
- Small pure functions (inputs in, one result out) are easy to test: call them and compare.
- A tiny `assert_eq` is enough for a smoke test. It prints PASS/FAIL and counts failures.
- Design rule: the function returns data on stdout and failure on status; the test checks both.

In [32]:
%%bash
# --- library ---
str_upper()   { printf '%s' "$1" | tr '[:lower:]' '[:upper:]'; }
str_repeat()  { local i out=; for ((i = 0; i < $2; i++)); do out+=$1; done; printf '%s' "$out"; }
int_max()     { local m=$1; shift; local n; for n in "$@"; do (( n > m )) && m=$n; done; printf '%s' "$m"; }
require_int() { case $1 in ''|*[!0-9]*) echo "not an integer: $1" >&2; return 1;; esac; }

# --- tiny test harness ---
T_FAIL=0
assert_eq() {   # assert_eq expected actual label
  if [ "$1" = "$2" ]; then echo "PASS $3"; else echo "FAIL $3: expected [$1] got [$2]"; T_FAIL=$((T_FAIL + 1)); fi
}
assert_eq "HELLO" "$(str_upper hello)" "upper"
assert_eq "ababab" "$(str_repeat ab 3)" "repeat"
assert_eq "9" "$(int_max 3 9 4)" "max"
require_int 42; assert_eq 0 $? "require_int ok"
require_int 4x2 2>/dev/null; assert_eq 1 $? "require_int rejects"
assert_eq "wrong" "$(str_upper abc)" "deliberate failure"
echo "failures: $T_FAIL"

PASS upper
PASS repeat
PASS max
PASS require_int ok
PASS require_int rejects
FAIL deliberate failure: expected [wrong] got [ABC]
failures: 1


## 21. Common traps, side by side
- Passing `$@` unquoted, `local x=$(...)`, forgetting `return` vs `echo`, and `$(f)` losing variable changes cover most function bugs.
- Capturing output also captures anything else the function prints, such as debug lines. Send those to stderr.

In [33]:
%%bash
# bad: unquoted "$@" splits a file name with a space
ls_bad()  { printf '[%s]\n' $@; }
ls_good() { printf '[%s]\n' "$@"; }
ls_bad  "my file.txt" b
ls_good "my file.txt" b
# bad: debug output ends up in the captured result
bad_result()  { echo "computing..."; echo 42; }
good_result() { echo "computing..." >&2; echo 42; }
echo "bad:  [$(bad_result)]"
echo "good: [$(good_result 2>/dev/null)]"
# bad: using return to hand back a big or non-numeric value
ret_big() { return 1000; }; ret_big; echo "return 1000 gives $?"

[my]
[file.txt]
[b]
[my file.txt]
[b]
bad:  [computing...
42]
good: [42]
return 1000 gives 232
